# Exploration et Visualisation des Données (EDA)

Ce notebook a pour objectif de comprendre le dataset des Near Earth Objects (NEOs) collecté depuis l'API NASA NeoWs.
Nous allons analyser la qualité des données, les distributions, les corrélations et détecter d'éventuels outliers.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuration des visualisations
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 8)

ModuleNotFoundError: No module named 'pandas'

## 1. Chargement des données

In [ ]:
dataset_path = '../data/dataset.csv'
df = pd.read_csv(dataset_path)

print(f"Dimensions du dataset : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head()

## 2. Aperçu et Qualité des données

In [ ]:
# Types de colonnes et valeurs manquantes
df.info()

In [ ]:
# Statistiques descriptives
df.describe().T

In [ ]:
# Vérification des valeurs nulles
null_counts = df.isnull().sum()
if null_counts.sum() == 0:
    print("Aucune valeur manquante détectée (l'imputation a probablement été faite lors de la collecte).")
else:
    print(null_counts[null_counts > 0])

## 3. Analyse de la Variable Cible

La variable cible est `is_potentially_hazardous`.

In [ ]:
target = 'is_potentially_hazardous'
plt.figure(figsize=(10, 7))
sns.set_theme(style="whitegrid")
palette = ["#3498db", "#e74c3c"]
# Correction ici : on ajoute hue=target et legend=False
ax = sns.countplot(x=target, data=df, hue=target, palette=palette, legend=False)
total = len(df)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height() / total)
    x = p.get_x() + p.get_width() / 2 - 0.05
    y = p.get_height() + (total * 0.01)
    ax.annotate(percentage, (x, y), size=12, fontweight='bold', color='black')
plt.title('Distribution de la Variable Cible (Déséquilibre des Classes)', fontsize=15, fontweight='bold')
plt.xlabel('Potentiellement Dangereux (0=Non, 1=Oui)', fontsize=12)
plt.ylabel('Nombre d\'objets', fontsize=12)
sns.despine()
plt.show()

## 4. Analyse des Distributions Numériques

In [ ]:
# Sélection des colonnes numériques principales
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != target]

# Visualisation de quelques distributions clés
cols_to_plot = ['absolute_magnitude_h', 'estimated_diameter_min_km', 'relative_velocity_km_per_second', 'miss_distance_astronomical']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
    if col in df.columns:
        sns.histplot(df[col], kde=True, ax=axes[i])
        axes[i].set_title(f'Distribution de {col}')

plt.tight_layout()
plt.show()

## 5. Analyse des Corrélations

In [ ]:
plt.figure(figsize=(14, 10))
corr_matrix = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Matrice de Corrélation')
plt.show()

# Top corrélations avec la cible
print("Corrélations avec la variable cible (is_potentially_hazardous) :")
print(corr_matrix[target].sort_values(ascending=False))

## 6. Détection des Outliers

Analyse via Boxplots pour identifier les valeurs extrêmes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
    if col in df.columns:
        sns.boxplot(x=target, y=col, data=df, ax=axes[i])
        axes[i].set_title(f'Boxplot de {col} par rapport à la cible')

plt.tight_layout()
plt.show()

## 7. Analyse des Variables Catégorielles

In [ ]:
cat_cols = ['orbit_class_type', 'orbiting_body']

for col in cat_cols:
    if col in df.columns:
        plt.figure(figsize=(10, 6))
        sns.countplot(y=col, hue=target, data=df)
        plt.title(f'Distribution de {col} par rapport à la cible')
        plt.show()

## Conclusions

- **Déséquilibre de classe** : Vérifier si le dataset est équilibré ou si des techniques de rééchantillonnage sont nécessaires.
- **Features discriminantes** : Identifier quelles variables (ex: magnitude, diamètre) semblent le plus influencer la dangerosité.
- **Prétraitement** : Décider s'il faut scaler les données ou gérer des outliers spécifiques avant la modélisation.